In [1]:
import random
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.options.display.max_columns = 100

import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import seaborn as sns
from collections import Counter
import json

import all_blocks
from merge_contracts import ContractsMerger
from generate_test import generate_preds
from get_distribution_utils import get_disrib_sums, cost_columns_to_datetime


c:\users\никита\appdata\local\programs\python\python39\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Подгружаем и обрабатываем датку

In [2]:
pays_df1 = pd.read_excel("data/Счета на оплату 3800-2023.XLSX")
pays_df2 = pd.read_excel("data/Счета на оплату 4200-4000-3800-2024.XLSX")
pays_df3 = pd.read_excel("data/Счета на оплату 5400-2023.XLSX")
pays_df4 = pd.read_excel("data/Счета на оплату 5400-2024.XLSX")
pays_df5 = pd.read_excel("data/Счета на оплату 5500-2023.XLSX")

In [3]:
pays_df = pd.concat([pays_df1, pays_df2, pays_df3, pays_df4, pays_df5], axis=0)

In [100]:
merger_df = pd.read_excel("data/Связь договор - здания.XLSX")
main_costs_df = pd.read_excel("data/Основные средства.XLSX")
squares = pd.read_excel("data/Площади зданий.XLSX")
serv_codes = pd.read_excel("data/Коды услуг.XLSX")



In [5]:
main_costs_df["тест_фича_нум"] = [random.randint(0, 100) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_бул"] = [random.randint(0, 1) for i in range(len(main_costs_df))]
main_costs_df["тест_фича_дата"] = ["19.04.2006"] * len(main_costs_df)

In [6]:
merger = ContractsMerger(pays_df, merger_df, main_costs_df, squares, serv_codes)

E:\DIR_python_projects\ledaer_digital_transformation_24\zakupai\services\api\ml\lib\merge_contracts.py:78: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.needed_pays_df["time"] = self.needed_pays_df["Год"].map(lambda x: datetime(year=x + 1, month=1, day=1))


In [7]:
import time

t = time.time()
res, features = merger.start_merging()
print(time.time() - t)

57.62276363372803


In [8]:
# сохраняем результат с 1 страницы загрузки данных
res.to_csv("res_datetimes.csv")
with open("features.pkl", "wb") as f:
    pickle.dump(features, f)

res остается лежать на бэке где-то
features нужны на фронте чтобы блоки с признаками создать и на бэке дальше
то есть там всегда будут 3 блока условия + блоки действий + feature из этого файла

## Подгружаем граф, запускаем распределение

In [184]:
features

[('тест_фича_бул', 'feature_custom_0', 'bool'),
 ('тест_фича_дата', 'feature_custom_1', 'date'),
 ('тест_фича_нум', 'feature_custom_2', 'num'),
 ('Признак "Используется в основной деятельности"',
  'feature_used_in_core',
  'bool'),
 ('Признак "Способ использования"', 'feature_usage_variants', 'bool'),
 ('Стоимость без НДС', 'feature_no_nds_cost', 'num'),
 ('Площадь ОС', 'feature_main_cost_square', 'num'),
 ('Площадь', 'feature_asset_area', 'num'),
 ('Дата ввода в эксплуатацию', 'feature_usage_start', 'date'),
 ('Дата начала действия связи с зданием', 'feature_start_using_date', 'date'),
 ('Начало владения', 'feature_start_private_date', 'date'),
 ('ID услуги', 'feature_service_id', 'num'),
 ('Дата окончания действия связи с зданием', 'feature_end_using_date', 'date'),
 ('Конец владения', 'feature_end_owning', 'date')]

In [179]:
features = pickle.load(open("features.pkl", "rb"))
res = pd.read_csv("res_datetimes.csv", index_col=0)

C:\Users\843E~1\AppData\Local\Temp/ipykernel_13564/930177580.py:2: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  res = pd.read_csv("res_datetimes.csv", index_col=0)


In [237]:
numeric_features = [i[0] for i in features if i[2] != "date"]
date_features = [i[0] for i in features if i[2] == "date"]

res = cost_columns_to_datetime(res, date_features)

In [246]:
res_by_prime = res.groupby("prime")
raw_graph = json.load(open("graphs/graph (6).json", "r"))

# TMP
# raw_graph["nodes"][2]["type"] = "use_feature_distribution"

In [247]:
all_blocks.init_graph(raw_graph, features)

In [248]:
unique_primes = res["prime"].unique()
distrib_sums = get_disrib_sums(res_by_prime, res["prime"].unique(), numeric_features)

100%|██████████| 4717/4717 [00:10<00:00, 460.52it/s]


In [249]:
preds = generate_preds(pays_df, serv_codes, res, distrib_sums)

100%|██████████| 4717/4717 [00:12<00:00, 367.49it/s]


In [244]:
preds.to_csv("Результат_распределенные_счета.csv")

In [250]:
preds[preds["Номер счета"] == 5006206451]

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения,Счет главной книги,ID услуги
84706,5500,2023,5006206451,1,0,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053849490,0,0,0.0,0.00,7048406010,S012
84707,5500,2023,5006206451,1,1,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053849491,0,0,0.0,0.00,7048406010,S012
84708,5500,2023,5006206451,1,2,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053849492,0,1,13.0,80.41,7048406010,S012
84709,5500,2023,5006206451,1,3,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053849493,0,1,1.0,6.19,7048406010,S012
84710,5500,2023,5006206451,1,4,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053968820,1,0,3368.5,20838.30,7048406010,S012
84711,5500,2023,5006206451,1,5,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053968821,0,1,1.0,6.19,7048406010,S012
84712,5500,2023,5006206451,1,6,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053968822,0,1,13.0,80.42,7048406010,S012
84713,5500,2023,5006206451,1,7,2023-02-09 00:00:00,ДПН 5500/163581,800006222,S012,ЗДН 5500/1/3871,60401018,55006040053968823,0,1,1.0,6.19,7048406010,S012


In [219]:
preds

,Компания,Год счета,Номер счета,Позиция счета,Номер позиции распределения,Дата отражения в учетной системе,ID договора,Услуга,Класс услуги,Здание,Класс ОС,ID основного средства,"Признак ""Использование в основной деятельности""","Признак ""Способ использования""",Площадь ОС,Сумма распределения,Счет главной книги,ID услуги
0,3800,2023,5006170938,1,0,2023-01-16 00:00:00,ДПН 3800/74661,800003299,S001,ЗДН 3800/1/265,60401018,38006040010385572,0,1,1.10,567713.35,7048208010,S001
1,3800,2023,5006170938,1,1,2023-01-16 00:00:00,ДПН 3800/74661,800003299,S001,ЗДН 3800/1/1538,60804001,38006080400345271,0,0,0.00,567713.35,7048208010,S001
56190,5400,2023,5006176256,1,0,2023-01-18 00:00:00,ДПН 5400/143136,800002266,S036,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.90,69554.00,7048414960,S036
56191,5400,2023,5006176256,2,0,2023-01-18 00:00:00,ДПН 5400/143136,800002266,S036,ЗДН 5400/1/7432,60804001,54006080400468090,1,0,150.90,51552.00,7048414960,S036
56192,5400,2023,5006197559,1,0,2023-02-01 00:00:00,ДПН 5400/143743,800006222,S012,ЗДН 5400/1/6824,60804001,54006080400468690,1,0,971.09,11796.80,7048406010,S012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56185,4200,2024,5006861096,1,82,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/6900,60401018,42006040054850570,1,0,5.53,261.83,7048209010,S004
56186,4200,2024,5006861096,1,83,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/7145,60804001,42006080400490040,1,0,30.70,261.83,7048209010,S004
56187,4200,2024,5006861096,1,84,2024-05-28 00:00:00,ДПН 4200/207590,800007377,S004,ЗДН 4200/1/7146,60804001,42006080400489420,1,0,11.25,261.83,7048209010,S004
56188,4200,2024,5006861117,1,0,2024-05-28 00:00:00,ДПН 4200/207349,800000706,S001,ЗДН 4200/1/423,60401018,42006040015620131,0,0,0.00,185594.89,7048208010,S001
